In [ ]:
##downoald the data

In [ ]:
#datasets = {}

#for time in range(0,5):
#    datasets[str(time)]={}
#    for c in ["change","non_change"]:
#        data = VBNDataset(file_path = file_path,dim =50,time_bin=0,change=c)
#        datasets[str(time)][c] = data

#import torch
#torch.save(datasets,"vbn.pth")
import torch


vbn_bins_5 = "https://drive.google.com/file/d/1HGwGm-iGVo7e573DMmp_NoFha1IaktVf/view?usp=sharing"



In [ ]:
import json
import os
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from src.libs.soi import SOI
from src.libs.util import get_samples
from src.vbn.vbn import VBNDataset
from experiments.config import get_config

parser = get_config()


parser.add_argument('--change', type=str, default="change", help ="options are change  or non_change stimuli")
parser.add_argument('--time_bin', type=int, default=0)


def benchmark_exp(args):

    
    r = {} 
    if args.setting == 0:
        structure = ['VISp', 'VISl', 'VISal']
    else:
        structure = ["VISp", "VISl", "VISal", "VISrl", "VISam", "VISpm" ]
    
    
   
    if args.arch == "mlp":
        total_dim = args.dim * len(structure)
        if total_dim <=30:
                hidden_dim = 128
        elif total_dim <=75:
                hidden_dim = 192
        elif total_dim <= 150:
                hidden_dim = 256
        else :
                hidden_dim = 128*3
        args.hidden_dim = hidden_dim
    else:
        args.hidden_dim = 120
        
    

    train_set = vbn_bins_5[bin][change]
    
    data_loader = DataLoader(train_set, batch_size=args.bs,shuffle=True, #pin_memory=True,
                                    num_workers=8, drop_last=True)

    device = "cuda" if args.accelerator == "gpu" else "cpu"
    
    test_samples = get_samples(
        data_loader, device=device, N=10000)
    
    model = SOI(args, nb_var=len(structure), var_list= {i: args.dim for i in structure})
 
    model.fit(data_loader, None)
    
    r = {"e": model.compute_o_inf(test_samples)}
    
    ## compute O_inf for each session/mouse
    sessions=train_set.get_sessions()
    r_s ={"o_inf":[],"s_inf":[],"tc":[],"dtc":[]}
    model.to(device)
    model.eval()
    for sess in sessions:
        out=model.compute_o_inf(sess)
        r_s["o_inf"].append(out["o_inf"])
        r_s["s_inf"].append(out["s_inf"])
        r_s["tc"].append(out["tc"])
        r_s["dtc"].append(out["dtc"])
    r["ses"]= r_s
    return r
        


if __name__ =="__main__":
    
    args = parser.parse_args()
    
    pl.seed_everything(args.seed)
    
    r = benchmark_exp(args)
    
    
    path = "{}/soi_vbn/{}/{}/{}/seed_{}/setting_{}/dim_{}/time_bin_{}/".format(args.results_dir, args.arch,
                                                               args.benchmark, args.transformation,
                                                               args.seed, args.setting, args.dim,args.time_bin)
    
    isExist = os.path.exists(path)
    if not os.path.exists(path):
        os.makedirs(path)
        
    path = path + "/results_{}.json".format(args.change)
    with open(path, 'w') as f:
        json.dump(r, f)

In [ ]:
## get all o-information value per session/mouse

## plot them using the scripts simialr to 
import os
import seaborn as sns
import matplotlib.pyplot as plt
import json
import pandas as pd
import seaborn as sns


def read_json(f):
    f = open(f) 
    return json.load(f) 

path_dir = "../../results/soi_vbn/{}/red/seed_{}/setting_{}/dim_{}/time_bin_{}/results_{}.json"

path = "plots_vbn/"
if not os.path.exists(path):
    os.makedirs(path)



def gen_plots_mice(bins,
              setting,
              seeds,
              arch ="mlp",
              path_dir = path_dir,
              dim = 20,
              met = "o_inf"):

    e= {met : {
        "change":[],
        "non_change":[]}  for met in bins}
    df=pd.DataFrame(columns=['bins',"seed", 'set', met,'mice'])
    
    for i, seed in enumerate( seeds ) :
        for bin in bins:
            file_path_change = path_dir.format(arch , seed,setting,dim,bin,"change")
            
            file_path_non_change = path_dir.format(arch , seed,setting,dim,bin,"non_change")
   
            out_change=read_json(file_path_change)
            out_non_change =read_json(file_path_non_change)

            change=out_change ["ses"][met]
            non_change=out_non_change ["ses"][met]
            e[bin]["change"].append(change)
            e[bin]["non_change"].append(non_change)
            for idx, (mice_change, mice_non_change) in enumerate( zip(change,non_change) ):
                df.loc[-1]= [bin,seed,"change",mice_change,idx ]
                df.index = df.index + 1  # shifting index
                df = df.sort_index()
                
                df.loc[-1]= [bin,seed,"non_change",mice_non_change,idx ]
                df.index = df.index + 1  # shifting index
                df = df.sort_index()    
    
    return df,e